# Boosting — Playground Series S6E4: Predicting Irrigation Need

XGBoost with Optuna tuning. Compares weighted vs unweighted. Reports both macro F1 and accuracy, tunes on F1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/Users/christopherli/kaggle_data/s6e4/train.csv')
test = pd.read_csv('/Users/christopherli/kaggle_data/s6e4/test.csv')
sample_sub = pd.read_csv('/Users/christopherli/kaggle_data/s6e4/sample_submission.csv')
print(train.shape, test.shape)

In [ ]:
# preprocessing - XGBoost can handle categoricals natively with enable_categorical=True
X = train.drop(columns=['id', 'Irrigation_Need'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

# convert strings to category dtype so XGBoost recognizes them
cat_cols = X.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

target_le = LabelEncoder()
y_enc = target_le.fit_transform(y)
print('Classes:', dict(zip(target_le.classes_, range(len(target_le.classes_)))))

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Baseline XGBoost

In [ ]:
xgb_base = XGBClassifier(
    n_estimators=300, random_state=42, enable_categorical=True,
    eval_metric='mlogloss', tree_method='hist', n_jobs=-1
)

cv_f1 = cross_val_score(xgb_base, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
cv_acc = cross_val_score(xgb_base, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

print(f'Baseline XGB (no weighting)')
print(f'  Macro F1: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'  Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})')

## Baseline XGBoost with sample weighting
XGBoost doesn't have a direct `class_weight` param for multiclass. We pass `sample_weight` during fit instead. To evaluate with CV, do it manually.

In [ ]:
f1_scores, acc_scores = [], []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]
    sw = compute_sample_weight(class_weight='balanced', y=y_tr)

    m = XGBClassifier(n_estimators=300, random_state=42, enable_categorical=True,
                      eval_metric='mlogloss', tree_method='hist', n_jobs=-1)
    m.fit(X_tr, y_tr, sample_weight=sw)
    p = m.predict(X_va)
    f1_scores.append(f1_score(y_va, p, average='macro'))
    acc_scores.append(accuracy_score(y_va, p))

print(f'Baseline XGB (sample_weight=balanced)')
print(f'  Macro F1: {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})')
print(f'  Accuracy: {np.mean(acc_scores):.4f} (+/- {np.std(acc_scores):.4f})')

## Optuna tuning (optimizing macro F1)

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10, log=True),
    }
    use_weights = trial.suggest_categorical('use_weights', [True, False])

    f1s = []
    for tr_idx, va_idx in skf.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]
        sw = compute_sample_weight('balanced', y_tr) if use_weights else None

        m = XGBClassifier(**params, random_state=42, enable_categorical=True,
                          eval_metric='mlogloss', tree_method='hist', n_jobs=-1)
        m.fit(X_tr, y_tr, sample_weight=sw)
        f1s.append(f1_score(y_va, m.predict(X_va), average='macro'))
    return np.mean(f1s)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best F1: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# evaluate on held-out val set
best_params = study.best_params.copy()
use_weights = best_params.pop('use_weights')

xgb_best = XGBClassifier(**best_params, random_state=42, enable_categorical=True,
                         eval_metric='mlogloss', tree_method='hist', n_jobs=-1)
sw = compute_sample_weight('balanced', y_train) if use_weights else None
xgb_best.fit(X_train, y_train, sample_weight=sw)
y_pred = xgb_best.predict(X_val)

print(f'Validation macro F1: {f1_score(y_val, y_pred, average="macro"):.4f}')
print(f'Validation accuracy: {accuracy_score(y_val, y_pred):.4f}')
print()
print(classification_report(y_val, y_pred, target_names=target_le.classes_))

In [ ]:
# feature importance
importances = pd.Series(xgb_best.feature_importances_, index=X.columns).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(9, 7), title='XGBoost Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# refit on all training data, predict test set, save submission
xgb_final = XGBClassifier(**best_params, random_state=42, enable_categorical=True,
                          eval_metric='mlogloss', tree_method='hist', n_jobs=-1)
sw = compute_sample_weight('balanced', y_enc) if use_weights else None
xgb_final.fit(X, y_enc, sample_weight=sw)
preds = xgb_final.predict(X_test)
preds_labels = target_le.inverse_transform(preds)

submission = sample_sub.copy()
submission['Irrigation_Need'] = preds_labels
submission.to_csv('../submissions/boosting_submission.csv', index=False)
print(submission.head())